In [4]:
import pandas as pd
from scipy.stats import spearmanr


In [ ]:
dataset = "impli"

sim_functions = [
    "cos",
    "cka",
    "wasser",
]

agg_functions = [
    "mean",
    "max",
    "sum",
    "entropy",
    "gini"
]


model_map = {
    "google-bert/bert-base-uncased": "BERT-base Uncased",
    "google-bert/bert-base-cased": "BERT-base Cased",
    "google-bert/bert-large-uncased": "BERT-large Uncased",
    "google-bert/bert-large-cased": "BERT-large Cased",
    "answerdotai/ModernBERT-base": "ModernBERT-base",
    "answerdotai/ModernBERT-large": "ModernBERT-large",
}

models = list(model_map.keys())

layer_map = {
    "google-bert/bert-base-uncased": list(range(12)),
    "google-bert/bert-base-cased": list(range(12)),
    "google-bert/bert-large-uncased": list(range(24)),
    "google-bert/bert-large-cased": list(range(24)),
    "answerdotai/ModernBERT-base": list(range(22)),
    "answerdotai/ModernBERT-large": list(range(28)),
}

# master file
# ROOT = Path.cwd().parent

# scores_dir = ROOT / "data" / "processed" 
csv_file = f"../../data/processed/impli_layers.csv"

print(csv_file)

df = pd.read_csv(csv_file)
df_human_decomp = pd.read_csv("/Users/mmi/Documents/projects/idioms_decomposability/decomp_code/idioms_decomposability/data/processed/bulkes_impli_human_decomp.csv")

print(f"{df_human_decomp.shape=}")

../../data/processed/impli_layers.csv
df_human_decomp.shape=(94, 3)


In [5]:
with_bulkes = df.merge(
    df_human_decomp,
    left_on="base_form",
    right_on="impli",
    how="inner"   # or "left", "right", "outer"
)

print(with_bulkes.shape)
print(with_bulkes.columns)
with_bulkes.head()


(247050, 21)
Index(['Unnamed: 0', 'premise', 'hypothesis', 'extracted_idiom', 'base_form',
       'idiom_pos', 'idiom_processed', 'token_scores', 'decomp_score',
       'sim_function', 'model_name', 'model_label', 'model_folder',
       'timestamp', 'layer', 'sim_func', 'agg_func', 'source_path', 'impli',
       'bulkes', 'PROPORTION RATED AS DECOMPOSABLE (%)'],
      dtype='object')


,Unnamed: 0,premise,hypothesis,extracted_idiom,base_form,idiom_pos,idiom_processed,token_scores,decomp_score,sim_function,...,model_label,model_folder,timestamp,layer,sim_func,agg_func,source_path,impli,bulkes,PROPORTION RATED AS DECOMPOSABLE (%)
0,1,"And then , the Daily Telegraph discovered ‘ th...","And then , the Daily Telegraph discovered ‘ th...",against the grain,against the grain,"[against, the, grain]",against the grain,"[(25, 'against', 0.015303194522857666), (26, '...",0.015303,cos,...,BERT-base Uncased,google-bert_bert-base-uncased,2025-12-25_02:34:13,0,cos,max,/Users/mmi/Documents/projects/idioms_decomposa...,against the grain,Go against the grain,0.4443
1,3,The imagination trembles at some of these idea...,The imagination trembles at some of these idea...,come clean,come clean,"[come, clean]",come clean,"[(19, 'come', 0.0060558319091796875), (20, 'cl...",0.006056,cos,...,BERT-base Uncased,google-bert_bert-base-uncased,2025-12-25_02:34:13,0,cos,max,/Users/mmi/Documents/projects/idioms_decomposa...,come clean,Come clean,0.4964
2,5,"All her insecurities surged painfully as , thr...","All her insecurities surged painfully as , reg...",throwing caution to the winds,throw caution to the wind,"[throw, caution, to, the, wind]",throwing caution to the winds,"[(10, 'throwing', 0.004351675510406494), (11, ...",0.008592,cos,...,BERT-base Uncased,google-bert_bert-base-uncased,2025-12-25_02:34:13,0,cos,max,/Users/mmi/Documents/projects/idioms_decomposa...,throw caution to the wind,Throw caution to the wind,0.3714
3,7,‘ It breaks my heart that his career has been ...,‘ It overwhelms me that his career has been ru...,breaks my heart,break someone's heart,"[break, someone, 's, heart]",breaks my heart,"[(3, 'breaks', -0.01068788766860962), (4, 'my'...",0.010688,cos,...,BERT-base Uncased,google-bert_bert-base-uncased,2025-12-25_02:34:13,0,cos,max,/Users/mmi/Documents/projects/idioms_decomposa...,break someone's heart,Break someone's heart,0.6421
4,8,For me it is quite out of the question that th...,For me it is impossible that the Maastricht tr...,out of the question,out of the question,"[out, of, the, question]",out of the question,"[(6, 'out', 0.01086956262588501), (7, 'of', 0....",0.012036,cos,...,BERT-base Uncased,google-bert_bert-base-uncased,2025-12-25_02:34:13,0,cos,max,/Users/mmi/Documents/projects/idioms_decomposa...,out of the question,Be out of the question,0.6849


In [22]:
def run_correlation_df(
    df,
    x_col="entropy_full",
    y_col="decomp_score"
):
    # drop NaNs just in case
    sub = df[[x_col, y_col]].dropna()

    if len(sub) < 3:
        return None

    corr, p_value = spearmanr(sub[x_col], sub[y_col])

    return {
        "spearmanr": corr,
        "p_value": p_value,
        "n": len(sub),
    }


group_cols = [
    "model_name",
    "layer",
    "sim_func",
    "agg_func",
    "timestamp",
]

results = []

for keys, group in with_bulkes.groupby(group_cols):
    stats = run_correlation_df(group, x_col="decomp_score", y_col="PROPORTION RATED AS DECOMPOSABLE (%)")

    if stats is None:
        continue

    record = dict(zip(group_cols, keys))
    record.update(stats)
    results.append(record)

human_decomp_corr = pd.DataFrame(results)


print(human_decomp_corr.shape)
human_decomp_corr.head()




(1830, 8)


,model_name,layer,sim_func,agg_func,timestamp,spearmanr,p_value,n
0,answerdotai/ModernBERT-base,0,cka,entropy,2025-12-25_02:34:31,0.092846,0.284143,135
1,answerdotai/ModernBERT-base,0,cka,gini,2025-12-25_02:34:31,0.085790,0.322496,135
2,answerdotai/ModernBERT-base,0,cka,max,2025-12-25_02:34:31,-0.005983,0.945091,135
3,answerdotai/ModernBERT-base,0,cka,mean,2025-12-25_02:34:31,-0.012742,0.883385,135
4,answerdotai/ModernBERT-base,0,cka,sum,2025-12-25_02:34:31,0.009922,0.909064,135


In [23]:
decomp_human_sig = human_decomp_corr[human_decomp_corr["p_value"] < 0.05]

decomp_human_sig.sort_values("spearmanr", ascending=False, inplace=True)

print(decomp_human_sig.shape)
decomp_human_sig.head(30)

(34, 8)


/var/folders/75/bf_qltnx6nq0w79rv4cx0ttc0000gn/T/ipykernel_55219/2169486243.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  decomp_human_sig.sort_values("spearmanr", ascending=False, inplace=True)


,model_name,layer,sim_func,agg_func,timestamp,spearmanr,p_value,n
1829,google-bert/bert-large-uncased,23,wasser,sum,2025-12-25_02:34:13,0.241241,0.004822,135
139,answerdotai/ModernBERT-base,9,cka,sum,2025-12-25_02:34:31,0.234678,0.006149,135
137,answerdotai/ModernBERT-base,9,cka,max,2025-12-25_02:34:31,0.221445,0.009847,135
138,answerdotai/ModernBERT-base,9,cka,mean,2025-12-25_02:34:31,0.220689,0.010108,135
124,answerdotai/ModernBERT-base,8,cka,sum,2025-12-25_02:34:31,0.218984,0.010718,135
109,answerdotai/ModernBERT-base,7,cka,sum,2025-12-25_02:34:31,0.218516,0.010891,135
1456,google-bert/bert-large-cased,23,cka,gini,2025-12-25_02:34:13,0.212328,0.013423,135
108,answerdotai/ModernBERT-base,7,cka,mean,2025-12-25_02:34:31,0.209640,0.014674,135
261,answerdotai/ModernBERT-base,17,cos,gini,2025-12-25_02:34:31,0.207764,0.015606,135
123,answerdotai/ModernBERT-base,8,cka,mean,2025-12-25_02:34:31,0.206566,0.016228,135


## just looking at the best config

In [21]:
subset = with_bulkes[
    (with_bulkes["model_name"] == "google-bert/bert-large-uncased") &
    (with_bulkes["layer"] == 23) &
    (with_bulkes["sim_func"] == "wasser") &
    (with_bulkes["agg_func"] == "sum")
]

print(subset.columns)

# print(subset.base_form.value_counts())
stats = run_correlation_df(
    subset,
    x_col="decomp_score",
    y_col="PROPORTION RATED AS DECOMPOSABLE (%)"
)

print("n rows in subset:", len(subset))
print(stats)


Index(['Unnamed: 0', 'premise', 'hypothesis', 'extracted_idiom', 'base_form',
       'idiom_pos', 'idiom_processed', 'token_scores', 'decomp_score',
       'sim_function', 'model_name', 'model_label', 'model_folder',
       'timestamp', 'layer', 'sim_func', 'agg_func', 'source_path', 'impli',
       'bulkes', 'PROPORTION RATED AS DECOMPOSABLE (%)'],
      dtype='object')
n rows in subset: 135
{'spearmanr': np.float64(0.24124133281509239), 'p_value': np.float64(0.004822446719863996), 'n': 135}


In [26]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

def bootstrap_spearman(df, col_x, col_y, n_boot=10_000, ci=95, seed=0):
    rng = np.random.default_rng(seed)

    x = df[col_x].to_numpy()
    y = df[col_y].to_numpy()

    # drop rows with NaNs in either column
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = len(x)
    if n < 3:
        raise ValueError("Need at least 3 valid paired observations for Spearman correlation.")

    boot_rho = np.empty(n_boot, dtype=float)

    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)  # sample row indices with replacement
        rho, _ = spearmanr(x[idx], y[idx])
        boot_rho[b] = rho

    alpha = (100 - ci) / 2
    lo, hi = np.percentile(boot_rho, [alpha, 100 - alpha])

    # point estimate on original data
    rho_hat, p_hat = spearmanr(x, y)

    return {
        "rho": rho_hat,
        "p_scipy": p_hat,
        "ci": (lo, hi),
        "boot_dist": boot_rho
    }


x_col="decomp_score"
y_col="PROPORTION RATED AS DECOMPOSABLE (%)"


# Example usage:
res = bootstrap_spearman(subset, x_col, y_col, n_boot=20000, ci=95, seed=42)
print(res["ci"])

(np.float64(0.0703321859981816), np.float64(0.39927374028095763))


In [ ]:
# print(len(set(subset.base_form.tolist())))

92
